# This is an interactive notebook to analyze trends in video game sales data from 2016 to make marketing decision in 2017

In [29]:
import pandas as pd 
import math as m
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats

In [30]:
df = pd.read_csv('games.csv')

In [31]:
display(df.head())

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


### Convert all columns to lowercase

In [32]:
df.columns = df.columns.str.lower()

In [33]:
display(df.head())

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [34]:
display(df.sample(10))

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
14292,Clannad,PS3,2011.0,Adventure,0.00,0.00,0.03,0.00,NaN,NaN,NaN
2177,Army of Two: The 40th Day,X360,2010.0,Shooter,0.62,0.24,0.00,0.09,73.0,7.3,M
16646,Time Travelers,PSP,2012.0,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
15826,Super Speed Machines,DS,2009.0,Racing,0.02,0.00,0.00,0.00,NaN,tbd,E
8007,NARC,XB,2005.0,Shooter,0.14,0.04,0.00,0.01,51.0,2.8,M
7296,Angry Birds Star Wars,PS4,2013.0,Strategy,0.10,0.08,0.00,0.04,47.0,2,E
9966,2 in 1 Combo Pack: Sonic Heroes / Super Monkey...,X360,2013.0,Misc,0.09,0.01,0.00,0.01,NaN,NaN,NaN
6732,Indiana Jones and the Staff of Kings,DS,2009.0,Action,0.16,0.07,0.00,0.02,50.0,6.8,T
10024,Final Fantasy III,PSP,2012.0,Role-Playing,0.00,0.00,0.11,0.00,NaN,8.9,T
9997,Deception IV: Blood Ties,PS3,2014.0,Action,0.03,0.02,0.06,0.01,70.0,8.2,M


In [35]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16713 non-null  object 
 1   platform         16715 non-null  object 
 2   year_of_release  16446 non-null  float64
 3   genre            16713 non-null  object 
 4   na_sales         16715 non-null  float64
 5   eu_sales         16715 non-null  float64
 6   jp_sales         16715 non-null  float64
 7   other_sales      16715 non-null  float64
 8   critic_score     8137 non-null   float64
 9   user_score       10014 non-null  object 
 10  rating           9949 non-null   object 
dtypes: float64(6), object(5)
memory usage: 1.4+ MB
None


### The only types I need to convert are year_of_release and user_score ... everything else seems to make sense 
##### making year_of relase an int made it cleaner and user_score being a float will be better for calculating later ... I wanted to convert to np.nan also so it was clear how many null values I have

In [36]:
# Change from floar to int
df['year_of_release'] = df['year_of_release'].astype('Int64')

# Change from object to float replace tbd and NaN with np.nan
df['user_score'] = df['user_score'].replace(['NaN', 'tbd'], np.nan)
df['user_score'] = df['user_score'].astype(float)

### Checking for missing values and duplicates 

In [37]:
print(df.isna().sum())
print(df.duplicated().sum())

name                  2
platform              0
year_of_release     269
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8578
user_score         9125
rating             6766
dtype: int64
0


### No fully duplicate rows but there is null values ... lots of missing values for both user and critic score and the rating ... and some missing year_of_release data
##### Could be from games not being popular enough for a review or from data collection issues

##### For both name and genre there is only two missing rows so I will just drop these rows
##### For rating I think there is too many missing to drop them and taking an average for a certain type might cause problems down the line so I think it's best to just replace them with Unknown

In [38]:
# drop 2 missing rows from name and genre
df = df.dropna(subset=['name', 'genre'])

# replace null with unknown in rating column
df['rating'] = df['rating'].fillna('Unknown')


### For Titles that repeat across platform I'm going to take the mode for year_of_release and apply it to the null values

In [ ]:
df['year_of_release'] = df.groupby('name')['year_of_release'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
